<a href="https://colab.research.google.com/github/yhuang-thu/FFmpeg/blob/master/convert_youtube_to_pptx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash
# 安装系统依赖
sudo apt-get update
sudo apt-get install -y tesseract-ocr libtesseract-dev

# 安装关键库
# yt-dlp: 代替 pytube，支持 YouTube/B站/腾讯视频等上千个网站
pip install yt-dlp easyocr python-pptx opencv-python pillow torch

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,628 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,442 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,870 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-dr

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


In [4]:
import cv2
import easyocr
import torch
import os
import io
import yt_dlp
from pptx import Presentation
from PIL import Image

# 1. 初始化 GPU OCR 引擎
device = "cuda" if torch.cuda.is_available() else "cpu"
reader = easyocr.Reader(['ch_sim', 'en'], gpu=(device == "cuda"))
print(f"✅ OCR 引擎已就绪，运行设备: {device}")

# --- 配置参数 ---
video_url = 'https://www.bilibili.com/video/BV1Bj411a7WS/?spm_id_from=333.337.search-card.all.click&vd_source=fb1cfade046b9173c24fbc79f7e67ec5' # 支持 B站 或 YouTube 链接
output_ppt = 'presentation_output.pptx'
sample_interval = 10  # 每 3 秒提取一帧（降低重复率）

def download_video(url):
    print(f"正在分析并下载视频: {url}")
    ydl_opts = {
        'format': 'mp4[height<=720]', # 限制 720p 足够 OCR，下载更快
        'outtmpl': 'downloaded_video.mp4',
        'quiet': True,
        'no_warnings': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    return 'downloaded_video.mp4'

def video_to_ppt_gpu(v_path, out_path, interval):
    cap = cv2.VideoCapture(v_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(fps * interval)

    ppt = Presentation()
    frame_count = 0
    slide_count = 0

    print("开始执行 GPU 加速识别...")

    while cap.isOpened():
        # 极速跳帧逻辑
        if frame_count % frame_interval == 0:
            ret, frame = cap.read()
            if not ret: break

            # GPU 加速识别
            results = reader.readtext(frame, detail=0)
            text_content = " ".join(results)

            # 内存图像处理 (不写磁盘)
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img_pil = Image.fromarray(frame_rgb)
            img_buf = io.BytesIO()
            img_pil.save(img_buf, format='JPEG', quality=80)
            img_buf.seek(0)

            # 插入 PPT
            slide = ppt.slides.add_slide(ppt.slide_layouts[5])
            slide.shapes.add_picture(img_buf, 0, 0, height=ppt.slide_height)

            # 插入识别文本框
            txBox = slide.shapes.add_textbox(0, ppt.slide_height - 100, ppt.slide_width, 100)
            txBox.text_frame.text = text_content if text_content.strip() else "[No text detected]"

            slide_count += 1
            if slide_count % 10 == 0:
                print(f"已处理 {slide_count} 页...")
        else:
            # 只抓取不解码，极速模式
            ret = cap.grab()
            if not ret: break
        frame_count += 1

    cap.release()
    ppt.save(out_path)
    print(f"🎉 处理完成！共生成 {slide_count} 页 PPT。")

# --- 执行主程序 ---
try:
    local_video = download_video(video_url)
    video_to_ppt_gpu(local_video, output_ppt, sample_interval)
    # 下载结果到本地
    from google.colab import files
    files.download(output_ppt)
except Exception as e:
    print(f"发生错误: {e}")

✅ OCR 引擎已就绪，运行设备: cuda
正在分析并下载视频: https://www.bilibili.com/video/BV1Bj411a7WS/?spm_id_from=333.337.search-card.all.click&vd_source=fb1cfade046b9173c24fbc79f7e67ec5
开始执行 GPU 加速识别...
已处理 10 页...
已处理 20 页...
已处理 30 页...
已处理 40 页...
已处理 50 页...
已处理 60 页...
已处理 70 页...
已处理 80 页...
已处理 90 页...
已处理 100 页...
已处理 110 页...
已处理 120 页...
已处理 130 页...
已处理 140 页...
已处理 150 页...
已处理 160 页...
已处理 170 页...
🎉 处理完成！共生成 177 页 PPT。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>